# Day 06 · 賦予行動力：Custom Tools 與 Function Tools

> 第二部・裝備升級　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 06 - 賦予行動力：Custom Tools 與 Function Tools.md`

## 今天要學會

1. 寫出同步、長時間執行、以及 `AgentTool` 三種形態的工具
2. 用 `BaseToolset` 打包一組工具
3. 讓工具平行執行
4. 避開「一個 agent 只能有一個內建工具」的限制

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. LLM 只會猜下一個字

這句話值得先具體化。模型沒有時鐘、沒有資料庫、不會算大數。
工具就是把這些能力接上去。

ADK 的工具分三大類：

| 類別 | 誰執行 | 例子 |
|---|---|---|
| **Function Tool** | 你的 Python 程序 | 查 DB、呼叫 API、算數學 |
| **Built-in Tool** | Google 端 | `google_search`、code execution |
| **Third-party / 協定** | 外部服務 | MCP、OpenAPI（Day 07） |

今天專注在第一類——它佔了你實際會寫的 95%。

## 2. 形態一：同步函式（最常見）

一個函式就是一個工具。ADK 從**型別註記**和 **docstring** 自動生成 schema。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool

_INVENTORY = {"A-100": 12, "A-200": 0, "B-300": 47}


def check_stock(sku: str) -> dict:
    """查詢商品的庫存數量。

    Args:
        sku: 商品編號，例如 'A-100'。
    """
    if sku not in _INVENTORY:
        return {"error": f"查無此商品：{sku}", "available_skus": list(_INVENTORY)}
    qty = _INVENTORY[sku]
    return {"sku": sku, "quantity": qty, "in_stock": qty > 0}


agent = LlmAgent(
    name="stock_agent",
    model=get_model(),
    instruction="你是庫存助理。一律呼叫 check_stock 查詢，用繁體中文簡短回答。",
    tools=[check_stock],
)

print(await run_once(agent, "A-200 還有貨嗎？", trace=True))

  🔧 [stock_agent] 呼叫 check_stock({'sku': 'A-200'})
  ↩️  [stock_agent] check_stock 回傳 {'sku': 'A-200', 'quantity': 0, 'in_stock': False}


  💬 [stock_agent] A-200 目前沒有庫存。
A-200 目前沒有庫存。


### 📌 看清楚模型收到什麼

這一步值得做一次。**docstring 是真的會被送出去的 API 文件**，不是註解。

In [3]:
import json

decl = FunctionTool(func=check_stock)._get_declaration()
print("name:", decl.name)
print("\ndescription:")
print(decl.description)
print("\nparameters_json_schema:")
print(json.dumps(decl.parameters_json_schema, indent=2, ensure_ascii=False))

name: check_stock

description:
查詢商品的庫存數量。

Args:
    sku: 商品編號，例如 'A-100'。

parameters_json_schema:
{
  "properties": {
    "sku": {
      "title": "Sku",
      "type": "string"
    }
  },
  "required": [
    "sku"
  ],
  "title": "check_stockParams",
  "type": "object"
}


對照可以發現：

- `Args:` 區塊 → 參數說明
- 型別註記 → `type`
- 沒有預設值的參數 → 進 `required`

所以「工具沒被呼叫」「參數傳錯」的第一個檢查點永遠是 docstring。

## 3. 形態二：`LongRunningFunctionTool`

有些工作要跑很久（產報表、跑批次、等外部審核）。讓 agent 卡在那裡等，
使用者只會看到畫面凍住。

`LongRunningFunctionTool` 的模型是：

> 函式**立刻回傳一個「已受理」的狀態**（工單號、預估時間），
> 真正的結果之後再由你的應用送回來。

In [4]:
from google.adk.tools import LongRunningFunctionTool


def request_report(month: str) -> dict:
    """排入一份營運報表的產製工作（需要數分鐘）。

    Args:
        month: 月份，格式 YYYY-MM。
    """
    # 立刻回傳受理狀態，不要在這裡等
    return {"status": "pending", "ticket": f"RPT-{month}", "eta_minutes": 5}


report_tool = LongRunningFunctionTool(func=request_report)
print("工具名稱      :", report_tool.name)
print("is_long_running:", report_tool.is_long_running)

工具名稱      : request_report
is_long_running: True


### 📌 ADK 會偷偷改你的工具說明

這是個沒什麼人提、但很有意思的細節——`LongRunningFunctionTool` 會在
description 後面**自動附加一段給模型的提醒**：

In [5]:
decl_long = report_tool._get_declaration()
print("你寫的 docstring 之後被接上了：\n")
print(decl_long.description[-150:])

你寫的 docstring 之後被接上了：

h: 月份，格式 YYYY-MM。

NOTE: This is a long-running operation. Do not call this tool again if it has already returned some intermediate or pending status.


這句 `Do not call this tool again if it has already returned some
intermediate or pending status` 是在防止模型看到「pending」就以為失敗、
然後一直重打。**你不用自己在 docstring 裡寫這件事。**

In [6]:
report_agent = LlmAgent(
    name="report_agent",
    model=get_model(),
    instruction="使用者要報表時呼叫 request_report，把工單號與預估時間告訴他。",
    tools=[report_tool],
)

print(await run_once(report_agent, "幫我產 2026-08 的營運報表", trace=True))

  🔧 [report_agent] 呼叫 request_report({'month': '2026-08'})
  ↩️  [report_agent] request_report 回傳 {'status': 'pending', 'ticket': 'RPT-2026-08', 'eta_minutes': 5}


  💬 [report_agent] 已經為您排入 2026-08 的營運報表產製工作！

* **工單號碼**：`RPT-2026-08`
* **預估時間**：約 5 分鐘

報表正在背景處理中，請稍候片刻。
已經為您排入 2026-08 的營運報表產製工作！

* **工單號碼**：`RPT-2026-08`
* **預估時間**：約 5 分鐘

報表正在背景處理中，請稍候片刻。


agent 立刻回覆工單號，不會卡住。**真正的結果之後再送回來**——
那個「送回來」的機制（用同一個 `function_call_id` 補一則 function_response）
就是 **Day 15 人機協作**與 **Day 24 Resume** 的底層。

### ⚠️ 一個會讓 notebook 永遠跑不完的陷阱

直覺上你可能會想用 **async generator** 來回報進度：

```python
async def generate_report(month: str):
    for step in steps:
        yield {"status": "pending", "step": step}   # 回報進度
    yield {"status": "done", "url": ...}
```

**這個寫法不會動，而且不會報錯。** 實測結果：

```
↩️  generate_report 回傳 {'result': <async_generator object at 0x...>}
```

ADK **不會替你迭代那個 generator**，它把 generator 物件本身當成結果丟給模型。
接著因為 `is_long_running=True`，整次執行就停在那裡等一個永遠不會來的完成訊號。

底下用一個受控的超時把這件事示範出來（不會真的卡住 notebook）：

In [7]:
import asyncio
from typing import Any, AsyncGenerator


async def gen_report(month: str) -> AsyncGenerator[dict[str, Any], None]:
    """❌ 錯誤示範：用 async generator 回報進度。

    Args:
        month: 月份。
    """
    yield {"status": "pending", "step": "撈資料"}
    yield {"status": "done", "month": month}


bad_agent = LlmAgent(
    name="bad_report",
    model=get_model(),
    instruction="使用者要報表時呼叫 gen_report。",
    tools=[LongRunningFunctionTool(func=gen_report)],
)

try:
    out = await asyncio.wait_for(
        run_once(bad_agent, "產 2026-08 報表", trace=True), timeout=40
    )
    print("結束:", out)
except asyncio.TimeoutError:
    print("\n⏸  40 秒後仍未結束——正如預期：")
    print("   generator 沒有被迭代，而 long-running 工具在等一個不會來的完成訊號。")


⏸  40 秒後仍未結束——正如預期：
   generator 沒有被迭代，而 long-running 工具在等一個不會來的完成訊號。


**記住**：`LongRunningFunctionTool` 的函式要**立刻 return 一個 dict**，
不要寫成 generator。進度回報要靠後續補送 function_response，不是靠 yield。

## 4. 形態三：`AgentTool` — 把 agent 當工具用

有時候你需要的「工具」其實是另一個 agent 的判斷力。

`AgentTool` 跟 `sub_agents` 的差別是**控制權**：

In [8]:
from google.adk.tools.agent_tool import AgentTool

translator = LlmAgent(
    name="translator",
    model=get_model(),
    description="把文字翻譯成指定語言。",
    instruction="把使用者給的文字翻成目標語言，只回譯文，不要解釋。",
)

writer = LlmAgent(
    name="writer",
    model=get_model(),
    instruction=(
        "你是文案。先用繁體中文寫一句標語，"
        "再用 translator 工具把它翻成英文，最後兩種都列出來。"
    ),
    tools=[AgentTool(agent=translator)],
)

print(await run_once(writer, "為一款登山背包寫標語", trace=True))

  🔧 [writer] 呼叫 translator({'request': '揹負夢想，征服每一座未知的山峰。'})


  ↩️  [writer] translator 回傳 {'result': 'Carry your dreams and conquer every unknown peak.'}


  💬 [writer] **中文標語：**
揹負夢想，征服每一座未知的山峰。

**英文標語：**
Carry your dreams and conquer every unknown peak.
**中文標語：**
揹負夢想，征服每一座未知的山峰。

**英文標語：**
Carry your dreams and conquer every unknown peak.


注意事件串流：`translator` 的呼叫長得就像一次普通的工具呼叫，
**控制權從來沒有離開 `writer`**。這是它跟 `sub_agents`（交棒）的根本差別。

| | `sub_agents` | `AgentTool` |
|---|---|---|
| 控制權 | 轉移出去 | 留在原地 |
| 後續對話 | 新 agent 接手 | 還是原本的 agent |
| 適合 | 分流到不同專員 | 「我需要一個子答案來完成工作」 |

## 5. `BaseToolset`：把一組工具打包

工具一多，`tools=[a, b, c, d, e, f, ...]` 會很難維護。
而且有些工具需要共用連線、需要在結束時清理資源。

`BaseToolset` 讓你把它們包成一個單位。

In [9]:
from google.adk.tools import BaseTool
from google.adk.tools.base_toolset import BaseToolset


class MathToolset(BaseToolset):
    """一組數學工具，共用同一份精度設定。"""

    def __init__(self, precision: int = 2):
        super().__init__()
        self.precision = precision
        self.call_log: list[str] = []

    async def get_tools(self, readonly_context=None) -> list[BaseTool]:
        def compound_interest(principal: float, rate: float, years: int) -> dict:
            """計算複利終值。

            Args:
                principal: 本金。
                rate: 年利率，例如 0.05 代表 5%。
                years: 年數。
            """
            self.call_log.append("compound_interest")
            value = principal * (1 + rate) ** years
            return {"final_value": round(value, self.precision)}

        def loan_payment(principal: float, annual_rate: float, months: int) -> dict:
            """計算每月房貸還款金額。

            Args:
                principal: 貸款金額。
                annual_rate: 年利率，例如 0.02 代表 2%。
                months: 還款月數。
            """
            self.call_log.append("loan_payment")
            r = annual_rate / 12
            if r == 0:
                return {"monthly_payment": round(principal / months, self.precision)}
            pay = principal * r / (1 - (1 + r) ** -months)
            return {"monthly_payment": round(pay, self.precision)}

        return [FunctionTool(func=compound_interest), FunctionTool(func=loan_payment)]

    async def close(self) -> None:
        """toolset 結束時被呼叫，用來關連線、清資源。"""
        self.call_log.append("closed")


math_tools = MathToolset(precision=0)

finance = LlmAgent(
    name="finance_agent",
    model=get_model(),
    instruction="你是理財顧問。一律呼叫工具計算，不要自己心算。用繁體中文回答。",
    tools=[math_tools],   # ← 整個 toolset 直接丟進去
)

print(await run_once(finance, "我借 800 萬、年利率 2%、分 360 期，每月要還多少？", trace=True))
print("\ntoolset 內部記錄:", math_tools.call_log)

  🔧 [finance_agent] 呼叫 loan_payment({'months': 360, 'principal': 8000000, 'annual_rate': 0.02})
  ↩️  [finance_agent] loan_payment 回傳 {'monthly_payment': 29570.0}


  💬 [finance_agent] 您好！根據您的條件（貸款金額 800 萬、年利率 2%、分 360 個月還款），使用本息平均攤還的方式計算，您每個月需要還款的金額為 **29,570** 元。
您好！根據您的條件（貸款金額 800 萬、年利率 2%、分 360 個月還款），使用本息平均攤還的方式計算，您每個月需要還款的金額為 **29,570** 元。

toolset 內部記錄: ['loan_payment']


`tools=[math_tools]` 一行就掛上兩個工具。而且它們共用 `self.precision`
這個設定——這是散裝函式做不到的。

## 6. 📌 補充：平行執行要「兩邊都做對」

原文提到工具可以平行執行（Python v1.10.0 起）。但有一件事文章沒強調：

> **只把工具寫成 async 是不夠的，prompt 也要寫成「可以平行」。**

如果你的 instruction 讓模型覺得必須一步一步來，它就會一次只發一個
function call，工具再怎麼 async 都沒用。實測給你看。

In [10]:
import time

CALL_TIMES: list[tuple[str, float]] = []


async def fetch_city_weather(city: str) -> dict:
    """查詢城市天氣（模擬一個 1 秒的網路請求）。

    Args:
        city: 城市名稱。
    """
    start = time.perf_counter()
    await asyncio.sleep(1.0)          # 模擬 I/O 等待
    CALL_TIMES.append((city, start))
    return {"city": city, "temp_c": 26, "condition": "多雲"}


# 版本 A：instruction 暗示要照順序做
serial_prompt = LlmAgent(
    name="serial_weather",
    model=get_model(),
    instruction=(
        "你是天氣助理。使用者問多個城市時，"
        "**一次查一個城市，查完一個再查下一個**，最後彙整。用繁體中文回答。"
    ),
    tools=[fetch_city_weather],
)

# 版本 B：instruction 明確允許一次全查
parallel_prompt = LlmAgent(
    name="parallel_weather",
    model=get_model(),
    instruction=(
        "你是天氣助理。使用者問多個城市時，"
        "**同時對所有城市發出查詢**（在同一輪一次呼叫多次工具），再一起彙整。"
        "用繁體中文回答。"
    ),
    tools=[fetch_city_weather],
)

question = "台北、東京、首爾現在天氣如何？"

for label, ag in (("A 一次一個", serial_prompt), ("B 一次全發", parallel_prompt)):
    CALL_TIMES.clear()
    t0 = time.perf_counter()
    await run_once(ag, question)
    elapsed = time.perf_counter() - t0
    # 各次工具呼叫的起始時間差，可以看出是不是同時發動的
    spread = max(t for _, t in CALL_TIMES) - min(t for _, t in CALL_TIMES) if CALL_TIMES else 0
    print(f"{label}: 總耗時 {elapsed:5.1f}s | 工具呼叫 {len(CALL_TIMES)} 次 | "
          f"起始時間差 {spread:.2f}s")

A 一次一個: 總耗時 189.9s | 工具呼叫 3 次 | 起始時間差 55.07s


B 一次全發: 總耗時  64.6s | 工具呼叫 3 次 | 起始時間差 0.00s


「起始時間差」接近 0 代表這幾次工具呼叫是**同時**發動的；
明顯大於 1 秒則代表它們是一輪一輪來的。

**結論**：工具寫成 `async` 只是讓平行「有可能」；
**instruction 才決定模型要不要一次發多個 function call**。兩邊都要做對。

### CPU-bound 的工具要卸載到執行緒

`async def` 只對 **I/O 等待**有用。如果你的工具是純計算，
`await` 不會讓出控制權，反而會把整個 event loop 卡住。

In [11]:
from concurrent.futures import ThreadPoolExecutor

_POOL = ThreadPoolExecutor(max_workers=4)


def _heavy(n: int) -> int:
    return sum(i * i for i in range(n))


async def crunch_numbers(n: int) -> dict:
    """做一段耗 CPU 的計算。

    Args:
        n: 計算規模。
    """
    loop = asyncio.get_running_loop()
    # ❌ 直接 return _heavy(n) 會卡住 event loop
    # ✅ 丟到執行緒池，讓其他協程還能跑
    result = await loop.run_in_executor(_POOL, _heavy, n)
    return {"n": n, "result": result}


t0 = time.perf_counter()
results = await asyncio.gather(*[crunch_numbers(2_000_000) for _ in range(4)])
print(f"4 個 CPU-bound 工具併發完成，耗時 {time.perf_counter() - t0:.2f}s")
print("結果一致:", len({r['result'] for r in results}) == 1)

4 個 CPU-bound 工具併發完成，耗時 0.24s
結果一致: True


## 7. ⚠️ 內建工具不能跟自訂工具混用

這是最常踩的限制之一。`google_search` 這類**由 Google 端執行**的內建工具，
不能跟你的函式工具放在同一個 agent。

In [12]:
from google.adk.tools import google_search

try:
    mixed = LlmAgent(
        name="mixed",
        model=get_model(),
        instruction="研究助理。",
        tools=[google_search, check_stock],
    )
    print(await run_once(mixed, "A-100 還有貨嗎？"))
except Exception as exc:
    msg = str(exc)
    print(f"{type(exc).__name__}:\n{msg[:400]}")
    if "429" in msg:
        print("\n（這次撞到的是 Search grounding 的獨立配額，不是混用限制本身。）")

_ResourceExhaustedError:

On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your cu

（這次撞到的是 Search grounding 的獨立配額，不是混用限制本身。）


### 解法：用 `AgentTool` 把它隔開

讓內建工具獨佔一個 agent，再把那個 agent 包成 `AgentTool`。
這樣主 agent 就能同時擁有「搜尋能力」和「自己的函式工具」。

In [13]:
searcher = LlmAgent(
    name="searcher",
    model=get_model(),
    description="用 Google 搜尋取得最新的公開資訊。",
    instruction="用搜尋找答案，用繁體中文摘要重點，三句話以內。",
    tools=[google_search],          # 獨佔，合法
)

assistant = LlmAgent(
    name="assistant",
    model=get_model(),
    instruction=(
        "你是助理。查庫存用 check_stock；需要外部最新資訊時用 searcher 工具。"
        "用繁體中文回答。"
    ),
    tools=[check_stock, AgentTool(agent=searcher)],   # 兩種能力共存
)

print("✅ 建立成功，工具清單:", [
    t.name if hasattr(t, "name") else getattr(t, "__name__", str(t))
    for t in assistant.tools
])
print(await run_once(assistant, "B-300 還有多少貨？", trace=True))

✅ 建立成功，工具清單: ['check_stock', 'searcher']


  🔧 [assistant] 呼叫 check_stock({'sku': 'B-300'})
  ↩️  [assistant] check_stock 回傳 {'sku': 'B-300', 'quantity': 47, 'in_stock': True}


  💬 [assistant] 商品 B-300 目前的庫存數量還有 47 個。
商品 B-300 目前的庫存數量還有 47 個。


## 8. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 模型不呼叫你的工具 | docstring 太模糊，或 instruction 沒要求「一定要用工具」 |
| 參數傳錯型別 | 少了型別註記，schema 就只是個 `object` |
| 工具明明 async 卻沒有變快 | instruction 讓模型一次只發一個呼叫——**prompt 也要改** |
| CPU-bound 工具拖垮整個服務 | `async def` 對純計算沒用，要 `run_in_executor` |
| 加了 `google_search` 之後其他工具全失效 | 內建工具不能混用，要用 `AgentTool` 隔開 |
| 工具要共用連線 / 要清理 | 用 `BaseToolset`，在 `close()` 裡收尾 |

## 9. 動手練習

1. 幫 `MathToolset` 加第三個工具（例如「投資報酬率」），
   確認 `tools=[math_tools]` 那行完全不用改。
2. 把第 6 節版本 B 的 instruction 改回「一次查一個」，
   確認「起始時間差」會從接近 0 變成 1 秒以上。
3. 把 `request_report` 的 `LongRunningFunctionTool` 換成普通的 `FunctionTool`，
   比較模型收到的 description 少了哪一句。
4. 寫一個 `BaseToolset`，在 `get_tools()` 裡依 `readonly_context` 的 state
   決定要回傳哪些工具（提示：管理員才看得到刪除工具）。

## 本日回顧

- **一個 Python 函式就是一個工具**；schema 來自型別註記 + docstring，
  **docstring 是送給模型的 API 文件**。
- **`LongRunningFunctionTool` 的函式要立刻 return 一個「已受理」狀態**，
  ⚠️ **寫成 async generator 不會動**（generator 不會被迭代，而且會卡住等完成訊號）。
  它會自動幫你在 description 附加「不要重複呼叫」的提醒。
  真正的完成訊號機制是 Day 15 / Day 24 的主題。
- **`AgentTool` 是外包不是交棒**，控制權留在原地。
- **`BaseToolset`** 打包一組共用設定／連線的工具，`close()` 負責收尾。
- **⚠️ 平行執行要兩邊都做對**：工具寫成 async **而且** instruction 要允許
  一次發多個呼叫。只改一邊沒有用。
- **⚠️ CPU-bound 要 `run_in_executor`**，`async def` 對純計算沒有幫助。
- **⚠️ 內建工具不能跟自訂工具混用**，用 `AgentTool` 隔開。

---
**下一天 → `../day07_mcp_and_openapi/`**